# NeRF Brain — Colab Training

Trains the neural-radiance-field on a NIfTI brain volume with checkpoint persistence to Google Drive.

**Runtime:** Colab Free T4 ≈ 1.5 h for the full 500-epoch run (early-stop usually around epoch 250–350).

## Before you run
1. **Runtime → Change runtime type → GPU** (T4 is fine on Free, A100 if you have Pro).
2. Run the cells top to bottom.
3. Don't close the tab — Colab Free disconnects after ~90 min idle. The checkpoint is flushed to Drive every 25 epochs, so a disconnect costs at most ~5 min of progress.

## 1. Environment check

In [ ]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Mount Google Drive
Checkpoints and TensorBoard logs are written to `/content/drive/MyDrive/nerf-brain/` so they survive disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/nerf-brain'
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/runs', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/brains', exist_ok=True)
print('Drive ready:', DRIVE_ROOT)

## 3. Clone repo and install deps

In [ ]:
!git clone https://github.com/TimurHegwein/Nerf---Neural-Radiance-Fields.git /content/nerf
%cd /content/nerf
!pip install -q nibabel nilearn scikit-image

## 4. Symlink Drive into the repo
We replace `checkpoints/`, `runs/`, `brains/` with symlinks to Drive — the training script writes to local paths, the bytes land on Drive.

In [ ]:
import os
for sub in ('checkpoints', 'runs', 'brains'):
    target = f'/content/drive/MyDrive/nerf-brain/{sub}'
    link = f'/content/nerf/{sub}'
    if os.path.exists(link) and not os.path.islink(link):
        import shutil; shutil.rmtree(link)
    if not os.path.exists(link):
        os.symlink(target, link)
    print(f'{link} -> {os.readlink(link)}')

## 5. Download the brain
Skipped automatically if `brains/brain_0.nii.gz` already exists on Drive.

In [ ]:
import os
if not os.path.exists('brains/brain_0.nii.gz'):
    !python fetch_brain.py
else:
    print('Brain already on Drive, skipping download.')

## 6. Start TensorBoard
Live PSNR / loss / LR curves. Refresh updates automatically while training runs in the next cell.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/

## 7. Run training

Streams live output. The current `main.py` config:
- `NeuralField(num_freqs=12, hidden_dim=512, num_layers=6)`
- `Adam lr=2e-3`, `ExponentialLR gamma=0.998`
- `RaySlabSampler n=8`, `batch_size=8192`
- `epochs=500`, `early_stop_patience=100`, `val_ratio=0.1` (stratified, seed=42)

On disconnect: re-run from this cell — the checkpoint on Drive holds the best state.

In [ ]:
!python main.py

## 8. Inspect the trained model
Loads the best checkpoint and renders a comparison dashboard inline.

In [ ]:
%matplotlib inline
import torch, numpy as np, matplotlib.pyplot as plt
from input.data import NiftiVolumeProvider
from output.renderer import NeuroRenderer
from output.visualizer import load_neural_field

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = load_neural_field('checkpoints/brain_0.pth', device=device)
ckpt = torch.load('checkpoints/brain_0.pth', map_location='cpu', weights_only=False)
print(f"Best Val Loss: {ckpt['best_val_loss']:.6f} @ epoch {ckpt['best_epoch']+1}")
print(f"Config: {ckpt['config']}")

provider = NiftiVolumeProvider('brains/brain_0.nii.gz', device=device)
renderer = NeuroRenderer(model, device=device)
renderer.plot_comparison(provider, num_slices=6, resolution=256)

## 9. Continuous Z-slicing
Renders 10 evenly spaced Z-positions to demonstrate the continuous representation.

In [ ]:
test_z = np.linspace(-1, 1, 10)
plt.figure(figsize=(20, 4))
for i, z in enumerate(test_z):
    img = renderer.render_slice(z_pos=float(z), resolution=128)
    plt.subplot(1, 10, i+1)
    plt.imshow(img, cmap='bone', vmin=0, vmax=1)
    plt.title(f'Z={z:.2f}')
    plt.axis('off')
plt.suptitle('Neural Field: Continuous Slicing')
plt.show()

## 10. Optional — 3D mesh extraction
Runs Marching Cubes on the implicit volume. Slow; lower resolution renders faster.

In [ ]:
from output.visualizer import NeuroVisualizer
viz = NeuroVisualizer('checkpoints/brain_0.pth', device=device)
viz.show(resolution=80, threshold=0.4)